# Employee Dataset Cleaning Project
This notebook documents the process of cleaning and preparing the Employee dataset for analysis.


## 1. Data Inspection and Initial Assessment
We begin by importing the necessary libraries (pandas, numpy) and loading the dataset.
This step helps us understand the structure, missing values, and data types.


In [1]:
import pandas as pd 
import numpy as np

### Loading the Dataset
We load the raw employee dataset into a pandas DataFrame for inspection.


In [3]:
df = pd.read_csv("Messy_Employee_dataset.csv")

### Initial Data Inspection

After loading the dataset, we perform a quick check to understand its structure, data types, missing values, and duplicate records.


In [4]:
df.head()

,Employee_ID,First_Name,Last_Name,Age,Department_Region,Status,Join_Date,Salary,Email,Phone,Performance_Score,Remote_Work
0,EMP1000,Bob,Davis,25.0,DevOps-California,Active,4/2/2021,59767.65,bob.davis@example.com,-1651623197,Average,True
1,EMP1001,Bob,Brown,NaN,Finance-Texas,Active,7/10/2020,65304.66,bob.brown@example.com,-1898471390,Excellent,True
2,EMP1002,Alice,Jones,NaN,Admin-Nevada,Pending,12/7/2023,88145.90,alice.jones@example.com,-5596363211,Good,True
3,EMP1003,Eva,Davis,25.0,Admin-Nevada,Inactive,11/27/2021,69450.99,eva.davis@example.com,-3476490784,Good,True
4,EMP1004,Frank,Williams,25.0,Cloud Tech-Florida,Active,1/5/2022,109324.61,frank.williams@example.com,-1586734256,Poor,False


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Employee_ID        1020 non-null   object 
 1   First_Name         1020 non-null   object 
 2   Last_Name          1020 non-null   object 
 3   Age                809 non-null    float64
 4   Department_Region  1020 non-null   object 
 5   Status             1020 non-null   object 
 6   Join_Date          1020 non-null   object 
 7   Salary             996 non-null    float64
 8   Email              1020 non-null   object 
 9   Phone              1020 non-null   int64  
 10  Performance_Score  1020 non-null   object 
 11  Remote_Work        1020 non-null   bool   
dtypes: bool(1), float64(2), int64(1), object(8)
memory usage: 88.8+ KB


In [7]:
df.isnull().sum()

Employee_ID            0
First_Name             0
Last_Name              0
Age                  211
Department_Region      0
Status                 0
Join_Date              0
Salary                24
Email                  0
Phone                  0
Performance_Score      0
Remote_Work            0
dtype: int64

In [8]:
df.duplicated().sum()

np.int64(0)

### Column-wise Observations

- **Column names**: Mixed case, some with underscores. Need to convert all to lowercase and strip whitespaces for consistency.
- **employee_id**: Clean, no issues.
- **first_name / last_name**: Clean, no issues.
- **age**: Currently float, should be integer. 211 missing values → replace with median.
- **department_region**: Contains combined info. Should be split by "-" for future analysis.
- **status**: Categorical. Check for inconsistencies.
- **join_date**: Stored as object. Needs conversion to proper datetime.
- **salary**: 24 missing values → replace with median.
- **email**: Mostly fine. Check for inconsistencies later.
- **phone**: Contains negative numbers → investigate further.
- **performance_score**: Categorical. Check for inconsistencies.
- **remote_work**: Boolean. Check for inconsistencies later.


## 2. Clean and Standardize Column Names

To ensure consistency across the dataset, all column names will be converted to lowercase and stripped of leading/trailing spaces. This makes the DataFrame easier to work with and avoids errors during analysis.


In [9]:
df.columns

Index(['Employee_ID', 'First_Name', 'Last_Name', 'Age', 'Department_Region',
       'Status', 'Join_Date', 'Salary', 'Email', 'Phone', 'Performance_Score',
       'Remote_Work'],
      dtype='object')

In [11]:
df.columns= df.columns.str.lower().str.strip()


## 3. Clean Whitespace in Text Columns

To ensure consistency, all text (string/object) columns will be stripped of leading and trailing whitespace.  
This prevents issues during analysis, such as mismatches in categorical values caused by hidden spaces.


In [14]:
for i in df.select_dtypes(include =["object"]).columns:
    df[i]=df[i].astype(str).str.strip()

## 4. Normalize Missing Values

Datasets often represent missing values inconsistently (e.g., "NA", "N/A", "null", empty strings).  
Since pandas only recognizes `np.nan` as a true missing value, we need to standardize all such variations to `np.nan`.  
This ensures consistent handling of missing data across the DataFrame.


In [19]:
na_values = ["na","NaN","null","NA",""]
for i in df.columns:
    if df[i].dtype=="object":
       df[i]=df[i].replace(to_replace= na_values,value=np.nan)

In [20]:
df.isnull().sum()

employee_id            0
first_name             0
last_name              0
age                  211
department_region      0
status                 0
join_date              0
salary                24
email                  0
phone                  0
performance_score      0
remote_work            0
dtype: int64

## 5. Standardize DateTime Columns

Standardizing date and time columns ensures they are stored in a consistent `datetime64[ns]` format.  
This provides several benefits:

- All date/time values follow a uniform internal representation.  
- Enables easy operations such as sorting, filtering, and extracting components (year, month, day).  
- Eliminates inconsistencies caused by mixed string formats in the dataset.


In [21]:
dates = [i for i in df.columns if any (j in i for j in["date","time","dt","timestamp"])]
for col in dates:
    try:
       df[col]=pd.to_datetime(df[col])
    except:
       pass

In [22]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   employee_id        1020 non-null   object        
 1   first_name         1020 non-null   object        
 2   last_name          1020 non-null   object        
 3   age                809 non-null    float64       
 4   department_region  1020 non-null   object        
 5   status             1020 non-null   object        
 6   join_date          1020 non-null   datetime64[ns]
 7   salary             996 non-null    float64       
 8   email              1020 non-null   object        
 9   phone              1020 non-null   int64         
 10  performance_score  1020 non-null   object        
 11  remote_work        1020 non-null   bool          
dtypes: bool(1), datetime64[ns](1), float64(2), int64(1), object(7)
memory usage: 88.8+ KB


## 6. Handle Missing Values (Numeric Columns)

Numeric columns often contain missing values that need to be standardized.  
Replacing them with the median ensures the dataset remains consistent without being skewed by extreme values.


In [23]:
for col in df.select_dtypes(include=['number']).columns:
    df[col]=df[col].fillna(df[col].median())
    

In [24]:
df.isnull().sum()

employee_id          0
first_name           0
last_name            0
age                  0
department_region    0
status               0
join_date            0
salary               0
email                0
phone                0
performance_score    0
remote_work          0
dtype: int64

## 7. Data Type Conversion

To ensure consistency and accurate analysis, columns must be stored in the correct data types.  
In this dataset, most columns are already in the proper format.  
Only the **age** column requires conversion from float to integer.


In [25]:
df['age']=df['age'].astype('int64')

In [26]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 12 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   employee_id        1020 non-null   object        
 1   first_name         1020 non-null   object        
 2   last_name          1020 non-null   object        
 3   age                1020 non-null   int64         
 4   department_region  1020 non-null   object        
 5   status             1020 non-null   object        
 6   join_date          1020 non-null   datetime64[ns]
 7   salary             1020 non-null   float64       
 8   email              1020 non-null   object        
 9   phone              1020 non-null   int64         
 10  performance_score  1020 non-null   object        
 11  remote_work        1020 non-null   bool          
dtypes: bool(1), datetime64[ns](1), float64(1), int64(2), object(7)
memory usage: 88.8+ KB


## 8. Inspect Categorical Columns

Before standardizing, it’s important to check the unique values in each categorical column.  
This helps identify inconsistencies such as case differences, extra spaces, or spelling variations.


In [28]:
categorical_cols = [ 'status','performance_score','remote_work']
for col in categorical_cols:
    print(f"{col}:",df[col].unique())

status: ['Active' 'Pending' 'Inactive']
performance_score: ['Average' 'Excellent' 'Good' 'Poor']
remote_work: [ True False]


## 

Observation:  
All categorical columns (`status`, `performance_score`, `remote_work`) are already consistent.  
No further standardization is required.


## 9. Split `department_region` Column

The `department_region` column contains two pieces of information combined into one string (e.g., `"HR_North"`).  
For better analysis, we will split this into two separate columns:
- **department** → the first part (e.g., `"HR"`)  
- **region** → the second part (e.g., `"North"`)


In [31]:
df[['department','region']]= df ['department_region'].str.split("-", expand = True)

In [32]:
df.head()

,employee_id,first_name,last_name,age,department_region,status,join_date,salary,email,phone,performance_score,remote_work,department,region
0,EMP1000,Bob,Davis,25,DevOps-California,Active,2021-04-02,59767.65,bob.davis@example.com,-1651623197,Average,True,DevOps,California
1,EMP1001,Bob,Brown,30,Finance-Texas,Active,2020-07-10,65304.66,bob.brown@example.com,-1898471390,Excellent,True,Finance,Texas
2,EMP1002,Alice,Jones,30,Admin-Nevada,Pending,2023-12-07,88145.90,alice.jones@example.com,-5596363211,Good,True,Admin,Nevada
3,EMP1003,Eva,Davis,25,Admin-Nevada,Inactive,2021-11-27,69450.99,eva.davis@example.com,-3476490784,Good,True,Admin,Nevada
4,EMP1004,Frank,Williams,25,Cloud Tech-Florida,Active,2022-01-05,109324.61,frank.williams@example.com,-1586734256,Poor,False,Cloud Tech,Florida


## 10. Inspect Phone Numbers
 
Phone numbers should never be negative.  
We will first inspect the `phone` column to confirm whether all values are negative.


In [35]:
all_negative = (df['phone'] < 0).all()
print("Are all phone numbers negative?",all_negative)
print(df['phone'].head())

Are all phone numbers negative? True
0   -1651623197
1   -1898471390
2   -5596363211
3   -3476490784
4   -1586734256
Name: phone, dtype: int64


#### Correct Negative Phone Numbers

Observation:  
Inspection showed thallll* phone numbers are stored as negative values (e.g., `-5596363211`).  
This is a consistent data entry error.  

Action:  
We will correct this by converting all phone numbers to their absolute values, ensuring they are positive and valid.


In [36]:
df['phone']= df['phone'].abs()

In [37]:
df['phone'].head()

0    1651623197
1    1898471390
2    5596363211
3    3476490784
4    1586734256
Name: phone, dtype: int64

## 11. Validation & Consistency Checks

Verifying the following:

- **Age range** → values fall within realistic employee limits (e.g., 18-65).  
- **Salary range** → values are non-negative and within expected business limits. (Since we don’t have business-specific salary thresholds, we will skip this check for limits)
- **Join date validity** → dates are valid calendar entries and not in the future.  
- **Category values** → categorical fields (`status`, `department`, etc.) contain only allowed values.

**Purpose:**  
To ensure the dataset is both internally consistent (no duplicates, no contradictions) and externally valid (follows real-world logic).  
This step confirms the data is trustworthy for analysis and reporting.

In [42]:
''' age range check 9 assuming employees should be between 18 to 65'''
invalid_age = df[(df['age']<18) | (df['age'] >65)]
print("Invalid ages:\n", invalid_age[['employee_id','age']])

Invalid ages:
 Empty DataFrame
Columns: [employee_id, age]
Index: []


In [49]:
'''Category values check'''
print("unique status values:", df['status'].unique())
print("performance_score values:",df['performance_score'].unique())
print("remote_work:",df['remote_work'].unique())


unique status values: ['Active' 'Pending' 'Inactive']
performance_score values: ['Average' 'Excellent' 'Good' 'Poor']
remote_work: [ True False]


In [50]:
'''Salary range check (non-negative)'''
invalid_salary = df[df['salary'] <0]
print("Invalid salaries:|n", invalid_salary[['employee_id','salary']])

Invalid salaries:|n Empty DataFrame
Columns: [employee_id, salary]
Index: []


In [52]:
'''Join date validity check (no future dates)'''
invalid_join_dates = df[df['join_date']> pd.Timestamp.today()]
print("invalid join dates:\n", invalid_join_dates[['employee_id','join_date']])


invalid join dates:
 Empty DataFrame
Columns: [employee_id, join_date]
Index: []


No invalid values detected after cleaning.

## Final Data Quality Review

In [53]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1020 entries, 0 to 1019
Data columns (total 14 columns):
 #   Column             Non-Null Count  Dtype         
---  ------             --------------  -----         
 0   employee_id        1020 non-null   object        
 1   first_name         1020 non-null   object        
 2   last_name          1020 non-null   object        
 3   age                1020 non-null   int64         
 4   department_region  1020 non-null   object        
 5   status             1020 non-null   object        
 6   join_date          1020 non-null   datetime64[ns]
 7   salary             1020 non-null   float64       
 8   email              1020 non-null   object        
 9   phone              1020 non-null   int64         
 10  performance_score  1020 non-null   object        
 11  remote_work        1020 non-null   bool          
 12  department         1020 non-null   object        
 13  region             1020 non-null   object        
dtypes: bool(

In [54]:
df.columns

Index(['employee_id', 'first_name', 'last_name', 'age', 'department_region',
       'status', 'join_date', 'salary', 'email', 'phone', 'performance_score',
       'remote_work', 'department', 'region'],
      dtype='object')

In [55]:
df.isnull().sum()

employee_id          0
first_name           0
last_name            0
age                  0
department_region    0
status               0
join_date            0
salary               0
email                0
phone                0
performance_score    0
remote_work          0
department           0
region               0
dtype: int64

In [56]:
df.describe()

,age,join_date,salary,phone
count,1020.000000,1020,1020.000000,1.020000e+03
mean,31.970588,2022-07-26 12:53:38.823529472,85164.299069,4.942253e+09
min,25.000000,2020-01-01 00:00:00,50047.320000,3.896086e+06
25%,30.000000,2021-04-12 06:00:00,68811.232500,2.520391e+09
50%,30.000000,2022-08-09 12:00:00,85547.870000,4.943997e+09
75%,35.000000,2023-10-31 12:00:00,100372.662500,7.341992e+09
max,40.000000,2024-12-29 00:00:00,119971.650000,9.994973e+09
std,5.136901,NaN,19638.385740,2.817326e+09


## Final Conclusion
The dataset is now:

- **Cleaned** → missing values handled, invalid entries corrected  
- **Standardized** → column names, data types, and formats aligned  
- **Structurally consistent** →  categories validated, logical ranges confirmed  
- **Ready for analysis** → reliable and trustworthy for further exploration, visualization, or modeling

## Export the final cleaned dataset to CSV


In [60]:
df.to_csv(r"C:\Users\garim\OneDrive\Desktop\github projects\data cleaning with python\Messy employee dataset\Cleaned_Employee_Dataset.csv", index=False)
